In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
from DeepScence.api import DeepScence
from tqdm import tqdm
from dca.api import dca
from SenCID.api import SenCID
import os
os.chdir(b'/Users/lele/Library/Mobile Documents/com~apple~CloudDocs/Research/Aging')

/Users/lele/Downloads/anaconda3/envs/sene/lib/python3.8/site-packages/kopt/config.py:60: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  _config = yaml.load(open(_config_path))


### read datasets and gs

In [ ]:
datasets = ["hayflick", "hca", "huvec", "notch", "oskm", "eto1"]
anchors = {
    "trans": ["STAT1", "Stat1"],
    "network": ["IL6", "Il6"],
    "sensig": [None, None],
    "Senmayo": [None, None],
    "geneAge": [None, None],
    "cellAge": [None, None],
    "CSgene": [None, None],
    "SASP": ["IL6", "Il6"],
    "Quest": [None, None]
}
all_gs = pd.read_csv("./data/coreGS_v2.csv", index_col=0)
columns_to_use = all_gs.columns[:9]
gs_list = {col: all_gs.index[all_gs[col] == True].tolist() for col in columns_to_use}

### 1. Collect all DeepScence scores (current pp)

In [ ]:
# do current

args = {
        "binarize": True,
        "verbose": False,
        "species": "human"
    }
for d in datasets:
    print(f"processing {d}...")
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VITRO/Current/h5ad/{d}_dca.h5ad")
    adata = adata[adata.obs["clean_label"]!="bad"]
    m = pd.DataFrame(index=adata.obs_names)
    m["y"] = adata.obs["SnC"].values
    
    # DeepScence + all gs
    for gsname, gs in gs_list.items():
        print(f"Running DeepScence + {gsname}: {len(gs)} genes...")
        anchor = anchors[gsname][0]
        adata = DeepScence(adata, custome_gs = gs, anchor_gene=anchor, **args)
        m[f"ds_{gsname}"] = adata.obs["ds"].values
        m[f"b_{gsname}"] = adata.obs["binary"].values

    # DeepScence + different n
    for n in [3,4,5,6]:
        print(f"Running DeepScence + >={n}...")
        adata = DeepScence(adata, n=n, **args)
        m[f"ds_{str(n)}+"] = adata.obs["ds"].values
        m[f"b_{str(n)}+"] = adata.obs["binary"].values

    # SenCID
    cid_scores = pd.read_csv(f"./data/VADLIATION_DATA/IN_VITRO/Current/metas/old/{d}_SenCID.csv", index_col=0)
    cid_scores = cid_scores.reindex(m.index)
    m["SID_score"] = cid_scores["SID_score"].values
    m["SID_binary"] = np.where(m["SID_score"] > 0.5, "SnC", "Normal")

    # save
    m.to_csv(f"./data/VADLIATION_DATA/IN_VITRO/Current/metas/{d}_scores_part2.csv")

### 2. Collect all DeepScence scores (Standard pp)

In [ ]:
# dca all
# for d in datasets:
#     print(f"processing {d}...")
#     adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VITRO/Standard/h5ad/{d}.h5ad")
#     adata.X = adata.raw.X
#     sc.pp.filter_genes(adata, min_cells=0.005 * adata.n_obs)
#     dca(adata, check_counts=False)
#     adata.write_h5ad(f"./data/VADLIATION_DATA/IN_VITRO/Standard/h5ad/{d}_dca.h5ad")

In [ ]:
# do standard
datasets = ["eto1"]
args = {
        "binarize": True,
        "verbose": False,
        "species": "human"
    }
for d in datasets:
    print(f"processing {d}...")
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VITRO/Standard/h5ad/{d}_dca.h5ad")
    # adata = adata[adata.obs["clean_label"]!="bad"]
    m = pd.DataFrame(index=adata.obs_names)
    m["y"] = adata.obs["SnC"].values


    # DeepScence + different n
    for n in [3,4,5,6]:
        print(f"Running DeepScence + >={n}...")
        adata = DeepScence(adata, n=n, **args)
        m[f"ds_{str(n)}+"] = adata.obs["ds"].values
        m[f"b_{str(n)}+"] = adata.obs["binary"].values
        
    # DeepScence + all gs
    for gsname, gs in gs_list.items():
        print(f"Running DeepScence + {gsname}: {len(gs)} genes...")
        anchor = anchors[gsname][0]
        adata = DeepScence(adata, custome_gs = gs, anchor_gene=anchor, **args)
        m[f"ds_{gsname}"] = adata.obs["ds"].values
        m[f"b_{gsname}"] = adata.obs["binary"].values


    # SenCID
    # adata.var['gene_symbols'] = adata.var.index
    # c = pd.read_csv("./data/in_vivo/gene_convert.csv")
    # gene_map = dict(zip(c['original'].dropna(), c['converted'].dropna()))
    # adata.var["converted_gene"] = adata.var["gene_symbols"].map(lambda x: gene_map.get(x, x))
    # adata.var_names = adata.var["converted_gene"].values
    # adata.var_names_make_unique()

    pred_dict, recSID, tmpfiles = SenCID(
                adata=adata,
                sidnums=[1, 2, 3, 4, 5, 6],
                denoising=False,
                binarize=True,
                threads=1,
                savetmp=True,
            )
    binary2 = []
    scores2 = []
    for i in range(len(recSID)):
        rec = recSID["RecSID"].iloc[i]
        score = pred_dict[rec]["SID_Score"].iloc[i]
        b = pred_dict[rec]["Binarization"].iloc[i]
        binary2.append(b)
        scores2.append(score)
    m["SID_binary"] = binary2
    m["SID_score"] = scores2
    
    # save
    m.to_csv(f"./data/VADLIATION_DATA/IN_VITRO/Standard/metas/{d}_scores_part2.csv")

In [4]:
import anndata

core = pd.read_csv("./data/coreGS_v2.csv", index_col=0)
core = core[core["n"]>=5].index.tolist()
core

expression_matrix = np.random.poisson(lam=1, size=(100, len(core)))

# Create an AnnData object
adata = anndata.AnnData(X=expression_matrix)
adata.obs_names = [f"Cell_{i}" for i in range(1, 101)]
adata.var_names = core

In [6]:
adata.write_h5ad("./data/additional_sc/fake_data.h5ad")